# L84 · MLOps 基础 — W&B 实验追踪、模型版本管理、Docker 打包与部署脚本

**目标**：把 Aurora 服务打包成可复现的 Docker 容器，并用 Weights & Biases 追踪训练实验的全部超参与指标。

🔗 Aurora 连接：`src/aurora/` 中的训练脚本 → Docker 镜像 → W&B Dashboard，实现从研究代码到可重现工程的最后一步。

研究代码在自己机器上跑通只是起点；真正的可重现性要求：任何人在任何机器上执行同一个镜像，能得到相同结果。
Docker 解决"环境漂移"问题，W&B 解决"实验失忆"问题——两者组合起来，让超参调优有据可查、模型版本有迹可循。

In [ ]:
# pip install wandb docker
import wandb

## 1. Dockerfile Best Practices：多阶段构建

**问题**：把开发依赖（pytest、black、jupyter）打进镜像会让镜像体积膨胀到 2 GB+，部署慢、攻击面大。

**解法**：多阶段构建（multi-stage build）——第一阶段 `builder` 安装所有依赖并编译，第二阶段 `runtime` 只从 `builder` 复制运行时需要的文件。

`.dockerignore` 排除 `__pycache__/`、`*.egg-info`、`.git/`、`notebooks/` 等，避免 build context 过大。

关键规则：
- `COPY pyproject.toml .` 先于 `COPY src/ src/`，利用层缓存——依赖不变时跳过 `pip install`
- 运行时用非 root 用户：`RUN useradd -m aurora && USER aurora`
- 固定基础镜像版本：`FROM python:3.11.9-slim` 而非 `python:latest`

In [ ]:
# 演示：打印一份最小可用的 Dockerfile 内容（不实际构建）
dockerfile = '''
# ── 阶段 1：builder ──────────────────────────────
FROM python:3.11.9-slim AS builder
WORKDIR /build
COPY pyproject.toml README.md ./
COPY src/ src/
RUN pip install --no-cache-dir --prefix=/install .[audio]

# ── 阶段 2：runtime ──────────────────────────────
FROM python:3.11.9-slim
COPY --from=builder /install /usr/local
RUN useradd -m aurora
USER aurora
ENTRYPOINT ["python", "-m", "aurora.serve"]
'''
print(dockerfile)

# .dockerignore 示例
dockerignore = '''
__pycache__/
*.egg-info/
.git/
notebooks/
*.pyc
.env
'''
print('--- .dockerignore ---')
print(dockerignore)

## 2. W&B 实验追踪：`wandb.init()` / `wandb.log()` / `wandb.save()`

`wandb.init(project='aurora-kws', config=cfg)` 开启一次 run，`config` 中填入所有超参（lr、batch_size、n_mels 等）。

训练循环中每步调用 `wandb.log({"loss": loss, "acc": acc, "step": step})`，W&B 自动绘制折线图。

训练结束调用 `wandb.save("model.pt")` 把权重上传到 run artifact，保证模型与超参一一对应。

三行核心代码：
```python
wandb.init(project='aurora-kws', config={'lr': 1e-3, 'batch_size': 32})
wandb.log({'loss': 0.42, 'val_acc': 0.91})
wandb.save('model.pt')
```

完整 run 结束后 `wandb.finish()` 关闭连接（Jupyter 中可以省略，脚本中建议显式调用）。

In [ ]:
# 演示：用 wandb 的 'disabled' 模式本地模拟一次训练 run（不需要联网）
import math, random

cfg = {'lr': 1e-3, 'batch_size': 32, 'epochs': 5, 'n_mels': 64}

run = wandb.init(
    project='aurora-kws-demo',
    config=cfg,
    mode='disabled',   # 离线演示，不实际上传
    name='demo-run'
)

# 模拟训练循环
for epoch in range(cfg['epochs']):
    loss = 1.0 * math.exp(-0.6 * epoch) + random.uniform(0, 0.05)
    val_acc = 1 - 0.5 * math.exp(-0.8 * epoch) + random.uniform(-0.02, 0.02)
    wandb.log({'epoch': epoch, 'train_loss': round(loss, 4), 'val_acc': round(val_acc, 4)})
    print(f'epoch={epoch}  loss={loss:.4f}  val_acc={val_acc:.4f}')

# 模拟保存模型
# wandb.save('model.pt')  # 联网时取消注释

wandb.finish()
print('✅ W&B run 完成（disabled 模式，仅本地演示）')

## 3. 实验追踪的价值：超参对比与最优配置定位

单次实验结果没有参考价值；**可比较的多次 run** 才能回答「lr=1e-3 比 lr=1e-4 好多少」。

W&B 的 **parallel coordinates plot** 把每组超参映射到一条折线，穿过最低 val_loss 区域的参数组合即为候选最优。

W&B **Sweep** 自动化这个过程：定义搜索空间 → Sweep Controller 分配任务 → 多个 agent 并行跑实验 → Dashboard 实时汇总。

对 Aurora KWS CNN 来说，最关键的两个轴是：
- `lr`：决定收敛速度与稳定性（搜索范围 `1e-4` 到 `1e-2`）
- `batch_size`：影响梯度噪声和训练时间（候选 `16 / 32 / 64`）

In [ ]:
# 演示：打印一份 W&B Sweep 配置（YAML 等价内容，用 dict 展示）
sweep_config = {
    'method': 'grid',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'lr': {
            'values': [1e-4, 1e-3, 1e-2]
        },
        'batch_size': {
            'values': [16, 32, 64]
        }
    }
}

import json as _json
print('sweep_config =')
print(_json.dumps(sweep_config, indent=2))
print()
print('grid search 共', len(sweep_config['parameters']['lr']['values'])
      * len(sweep_config['parameters']['batch_size']['values']), '组实验')

# 联网环境下执行 Sweep 的方式：
# sweep_id = wandb.sweep(sweep_config, project='aurora-kws')
# wandb.agent(sweep_id, function=train_fn, count=9)
print('✅ Sweep 配置打印完成（联网时取消注释执行）')

## 参数实验：`lr` × `batch_size` Grid Search

**搜索空间**：
- `lr` ∈ `{1e-4, 1e-3, 1e-2}` — 3 个学习率
- `batch_size` ∈ `{16, 32, 64}` — 3 个批大小
共 9 组实验，全部 grid 遍历。

**预期现象**：
- `lr=1e-2` + `batch_size=16`：梯度噪声大 + 学习率偏高，loss 震荡，val_acc 通常最低
- `lr=1e-3` + `batch_size=32`：经验最优区间，val_acc 峰值最高
- `lr=1e-4` + `batch_size=64`：收敛最慢，5 epoch 内 val_acc 还未到顶

W&B Dashboard 的 **parallel coordinates plot** 中，穿过高 `val_acc` 区域的折线集中在 `lr≈1e-3`，可视化地验证经验规律。

In [ ]:
# 本地模拟 9 组 grid search，观察 val_acc 分布
import math, random, itertools

lrs = [1e-4, 1e-3, 1e-2]
batch_sizes = [16, 32, 64]
EPOCHS = 5

results = []

for lr, bs in itertools.product(lrs, batch_sizes):
    random.seed(int(lr * 1e5) + bs)   # 固定种子保证可复现
    noise_scale = 0.05 / (bs ** 0.5)  # 大 batch 噪声小
    lr_penalty = abs(math.log10(lr) + 3) * 0.06  # 偏离 1e-3 越远惩罚越大
    final_acc = 0.91 - lr_penalty + random.uniform(-noise_scale, noise_scale)
    final_acc = round(min(max(final_acc, 0.6), 0.99), 4)
    results.append((lr, bs, final_acc))

# 按 val_acc 降序打印
results.sort(key=lambda x: -x[2])
print(f"{'lr':>8}  {'batch_size':>10}  {'val_acc':>8}")
print('-' * 34)
for lr, bs, acc in results:
    marker = ' ← 最优' if acc == results[0][2] else ''
    print(f"{lr:>8.1e}  {bs:>10}  {acc:>8.4f}{marker}")

print()
print('✅ Grid search 模拟完成')
print(f'最优配置：lr={results[0][0]:.1e}, batch_size={results[0][1]}, val_acc={results[0][2]}')

## 本课小结

本节通过 `wandb.init()` / `wandb.log()` / `wandb.save()` 将每次训练的超参与指标实时写入 W&B，
使 Aurora KWS CNN 的 9 组 `lr x batch_size` grid search 产出可对比的 `val_acc` 曲线。
Dockerfile 多阶段构建把 Aurora 服务封装为可移植镜像，`.dockerignore` 控制 build context 体积，确保部署环境与开发环境严格一致。
这两项能力共同对应 `src/aurora/` 训练流程的生产化，是 Aurora 从研究代码走向可重现工程的最后一步。
下一节（M6-I3）将介绍持续集成：用 GitHub Actions 自动运行 `make lint test`，并在每次 push 后触发 Docker 镜像重建。